# Databricks/PySpark dla Data Analyst/Engineer — Moduł 9: Optymalizacja i wydajność

Przez cały kurs wspominaliśmy o Catalyst, planach wykonania i kosztach shuffle'a --
ten moduł zbiera to wszystko w jedną całość: jak Spark optymalizuje zapytania, kiedy
warto cache'ować dane, jak kontrolować liczbę partycji, i jak czytać `.explain()` na
poważnie.

## Spis treści
1. [Catalyst i Tungsten: jak Spark optymalizuje zapytania](#sec1)
2. [cache() i persist()](#sec2)
3. [Partycjonowanie w pamięci: repartition kontra coalesce](#sec3)
4. [Dokładne czytanie planów wykonania](#sec4)
5. [Podsumowanie i ćwiczenia](#sec5)


Uruchamiamy (albo pobieramy już istniejącą) `SparkSession` — punkt wejścia do wszystkiego w Sparku. W Databricks `spark` jest już gotowe automatycznie w każdym notebooku; ten sam kod działa też lokalnie.

In [ ]:
from pyspark.sql import SparkSession

# Dodatkowe opcje JVM (--add-opens) sa potrzebne, gdy uruchamiasz Sparka lokalnie na
# Javie 17+ i chcesz uzywac pandas UDF / Apache Arrow (Modul 8) -- nowsze JDK domyslnie
# blokuja refleksyjny dostep do pamieci poza stosem, ktorego potrzebuje Arrow. W
# Databricks nie trzeba tego ustawiac recznie -- platforma robi to za Ciebie.
_opcje_jvm = (
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED"
)

spark = (
    SparkSession.builder.appName("kurs_spark")
    .config("spark.driver.extraJavaOptions", _opcje_jvm)
    .config("spark.executor.extraJavaOptions", _opcje_jvm)
    .getOrCreate()
)
spark


Wczytujemy nasz wspólny zestaw danych — dane sklepu internetowego (`klienci`, `produkty`, `pracownicy`, `zamowienia`, `pozycje_zamowien`), dokładnie ten sam schemat co w kursie SQL, ale tym razem jako DataFrame'y Sparka.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- klienci -----------------------------------------------------------
n_klienci = 300
miasta = ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segmenty = ["Nowy", "Standardowy", "Premium"]
prawdop_segmentu = [0.35, 0.45, 0.20]

klienci_pd = pd.DataFrame({
    "klient_id": np.arange(1, n_klienci + 1),
    "miasto": rng.choice(miasta, size=n_klienci, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segmenty, size=n_klienci, p=prawdop_segmentu),
    "dni_od_rejestracji": rng.integers(1, 900, size=n_klienci),
})

# --- produkty ------------------------------------------------------------
n_produkty = 40
kategorie = ["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]
srednie_ceny = {"Elektronika": 1200, "Odziez": 150, "Dom": 250, "Sport": 300, "Ksiazki": 50}

kategorie_produktow = rng.choice(kategorie, size=n_produkty)
produkty_pd = pd.DataFrame({
    "produkt_id": np.arange(1, n_produkty + 1),
    "kategoria": kategorie_produktow,
    "cena": [
        round(max(5.0, rng.normal(srednie_ceny[k], srednie_ceny[k] * 0.3)), 2)
        for k in kategorie_produktow
    ],
})

# --- pracownicy ----------------------------------------------------------
n_pracownicy = 12
regiony = ["Polnoc", "Poludnie", "Wschod", "Zachod"]
pracownicy_pd = pd.DataFrame({
    "pracownik_id": np.arange(1, n_pracownicy + 1),
    "region": rng.choice(regiony, size=n_pracownicy),
    "dzial": rng.choice(["Sprzedaz", "Obsluga"], size=n_pracownicy, p=[0.6, 0.4]),
})

# --- zamowienia ------------------------------------------------------------
n_zamowienia = 3000
klient_id_zamowien = rng.choice(klienci_pd["klient_id"], size=n_zamowienia)
pracownik_id_zamowien = rng.choice(pracownicy_pd["pracownik_id"], size=n_zamowienia)

segment_mapa = klienci_pd.set_index("klient_id")["segment"]
segment_zamowien = klient_id_zamowien.astype(object)
segment_zamowien = pd.Series(klient_id_zamowien).map(segment_mapa).values

# prawdopodobienstwo anulowania zalezy od segmentu klienta (celowo wbudowany efekt)
prawdop_anulowania = np.where(
    segment_zamowien == "Nowy", 0.22,
    np.where(segment_zamowien == "Standardowy", 0.10, 0.04),
)
anulowane = rng.binomial(1, prawdop_anulowania)
status_zamowien = np.where(
    anulowane == 1, "Anulowane",
    rng.choice(["Zlozone", "Wyslane", "Dostarczone"], size=n_zamowienia, p=[0.2, 0.3, 0.5]),
)

data_bazowa = pd.Timestamp("2024-01-01")
dni_offset = rng.integers(0, 730, size=n_zamowienia)

zamowienia_pd = pd.DataFrame({
    "zamowienie_id": np.arange(1, n_zamowienia + 1),
    "klient_id": klient_id_zamowien,
    "pracownik_id": pracownik_id_zamowien,
    "data_zamowienia": [data_bazowa + pd.Timedelta(days=int(d)) for d in dni_offset],
    "status": status_zamowien,
})

# --- pozycje_zamowien ------------------------------------------------------
wiersze_pozycji = []
pozycja_id = 1
cena_produktu = produkty_pd.set_index("produkt_id")["cena"]

for zamowienie_id in zamowienia_pd["zamowienie_id"]:
    liczba_pozycji = rng.integers(1, 5)
    wybrane_produkty = rng.choice(produkty_pd["produkt_id"], size=liczba_pozycji, replace=False)
    for produkt_id in wybrane_produkty:
        ilosc = int(rng.integers(1, 4))
        rabat = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        wiersze_pozycji.append({
            "pozycja_id": pozycja_id,
            "zamowienie_id": int(zamowienie_id),
            "produkt_id": int(produkt_id),
            "ilosc": ilosc,
            "cena_jednostkowa": float(cena_produktu[produkt_id]),
            "rabat": float(rabat),
        })
        pozycja_id += 1

pozycje_zamowien_pd = pd.DataFrame(wiersze_pozycji)

# --- konwersja na DataFrame'y Sparka ---------------------------------------
klienci = spark.createDataFrame(klienci_pd)
produkty = spark.createDataFrame(produkty_pd)
pracownicy = spark.createDataFrame(pracownicy_pd)
zamowienia = spark.createDataFrame(zamowienia_pd)
pozycje_zamowien = spark.createDataFrame(pozycje_zamowien_pd)

klienci.show(5)


<a id="sec1"></a>
## 1. Catalyst i Tungsten: jak Spark optymalizuje zapytania

Wspominaliśmy o nich od Modułu 1 -- czas zebrać to w jedną całość. Gdy budujesz łańcuch
transformacji, Spark NIE wykonuje go krok po kroku tak, jak go napisałeś. Zamiast tego:

1. **Plan logiczny** -- Spark tłumaczy Twój kod (DataFrame API albo SQL) na abstrakcyjne
   drzewo operacji, niezależnie od tego, jak je zapisałeś.
2. **Optymalizacja przez Catalyst** -- reguły optymalizujące przekształcają ten plan:
   łączenie kolejnych filtrów w jeden, *predicate pushdown* (przesuwanie filtrów jak
   najbliżej źródła danych), *column pruning* (czytanie tylko potrzebnych kolumn),
   wybór strategii joina (broadcast kontra shuffle, Moduł 4) i wiele innych.
3. **Plan fizyczny** -- zoptymalizowany plan logiczny zamienia się w konkretne operacje
   do wykonania na klastrze.
4. **Tungsten** -- silnik wykonawczy niższego poziomu, zarządzający pamięcią i
   generujący kod bajtowy JVM bezpośrednio dla planu fizycznego (*whole-stage code
   generation*), zamiast interpretować go krok po kroku.

> ⚡ **Dlatego SQL i DataFrame API dają identyczną wydajność**
>
> Niezależnie, czy piszesz `df.filter(...).select(...)` czy `SELECT ... WHERE ...` (Moduł 5), OBA trafiają do TEGO SAMEGO planu logicznego, przechodzą przez TE SAME reguły Catalysta, i kończą jako TEN SAM plan fizyczny. Wybór stylu to kwestia czytelności kodu, nigdy wydajności -- wspominaliśmy o tym już w Module 5.

In [ ]:
from pyspark.sql import functions as F

# Dwa rozne filtry napisane osobno -- Catalyst i tak polaczy je w jeden krok
plan = zamowienia.filter(F.col("status") == "Dostarczone").filter(
    F.col("klient_id") > 100
).select("zamowienie_id", "klient_id")
plan.explain()


<a id="sec2"></a>
## 2. cache() i persist()

Ponieważ Spark stosuje leniwą ewaluację (Moduł 1), za KAŻDYM razem, gdy wywołasz akcję na
DataFrame, cały łańcuch transformacji liczy się OD NOWA -- łącznie z ponownym wczytaniem
danych źródłowych. Jeśli używasz tego samego (kosztownego w policzeniu) DataFrame
WIELOKROTNIE, `cache()`/`persist()` zapamiętuje wynik po pierwszym wyliczeniu.

In [ ]:
zamowienia_z_wartoscia = zamowienia.join(pozycje_zamowien, "zamowienie_id").groupBy(
    "zamowienie_id", "klient_id"
).agg(F.sum(F.col("ilosc") * F.col("cena_jednostkowa") * (1 - F.col("rabat"))).alias("wartosc"))

zamowienia_z_wartoscia.cache()

# Pierwsza akcja -- faktycznie liczy caly lancuch I zapisuje wynik w pamieci
print(f"Liczba wierszy: {zamowienia_z_wartoscia.count()}")

# Kolejne akcje na tym samym DataFrame -- czytaja juz z cache'u, bez przeliczania od nowa
zamowienia_z_wartoscia.filter(F.col("wartosc") > 1000).show(5)


> ⚠️ **cache() bez pierwszej akcji nic nie robi**
>
> `.cache()` (jak każda transformacja) jest LENIWY -- sam w sobie tylko OZNACZA DataFrame jako 'do zapamiętania'. Rzeczywiste liczenie i zapisanie w pamięci następuje dopiero przy PIERWSZEJ akcji wywołanej PO `cache()`. Zapomnienie o tym to częsty błąd: ktoś woła `cache()`, a potem dziwi się, że kolejne uruchomienie wciąż jest wolne, bo tak naprawdę nigdy nie wykonał żadnej akcji, żeby cache faktycznie się wypełnił.

In [ ]:
# persist() z jawnym poziomem -- wiecej kontroli niz cache() (ktore to po prostu MEMORY_AND_DISK)
from pyspark import StorageLevel

zamowienia_z_wartoscia.unpersist()
zamowienia_z_wartoscia.persist(StorageLevel.MEMORY_ONLY)
zamowienia_z_wartoscia.count()
print(zamowienia_z_wartoscia.storageLevel)


> 🧱 **Kiedy WARTO cache'ować, a kiedy to strata pamięci**
>
> Cache ma sens, gdy DataFrame jest używany WIELOKROTNIE w dalszych krokach (np. kilka różnych agregacji na tym samym przefiltrowanym zbiorze) -- oszczędzasz powtarzanie kosztownych obliczeń. Cache'owanie DataFrame'u użytego TYLKO RAZ to czysta strata pamięci klastra. Pamiętaj też, żeby wołać `.unpersist()`, gdy dany DataFrame nie jest już potrzebny -- inaczej zajmuje pamięć niepotrzebnie przez resztę sesji.

<a id="sec3"></a>
## 3. Partycjonowanie w pamięci: repartition kontra coalesce

Liczba partycji DataFrame'u w pamięci decyduje o tym, ile zadań równoległych Spark może
uruchomić naraz. `repartition(n)` przetasowuje dane na DOKŁADNIE `n` partycji (zawsze
robi shuffle, może zarówno zwiększyć, jak i zmniejszyć liczbę partycji).
`coalesce(n)` ZMNIEJSZA liczbę partycji BEZ pełnego shuffle'a (łączy sąsiednie partycje
lokalnie) -- szybsze, ale działa tylko w jedną stronę (zmniejszanie).

In [ ]:
print(f"Liczba partycji przed: {zamowienia.rdd.getNumPartitions()}")

zamowienia_wiecej_partycji = zamowienia.repartition(8)
print(f"Po repartition(8): {zamowienia_wiecej_partycji.rdd.getNumPartitions()}")

zamowienia_mniej_partycji = zamowienia_wiecej_partycji.coalesce(2)
print(f"Po coalesce(2): {zamowienia_mniej_partycji.rdd.getNumPartitions()}")


In [ ]:
# repartition po kolumnie -- przydatne przed kosztownymi operacjami grupujacymi/joinujacymi
# na tej samej kolumnie, zeby dane z tym samym kluczem trafily do tej samej partycji
zamowienia_wg_klienta = zamowienia.repartition(4, "klient_id")
print(f"Liczba partycji: {zamowienia_wg_klienta.rdd.getNumPartitions()}")


> ⚠️ **Zbyt mało ANI zbyt wiele partycji to problem**
>
> Za MAŁO partycji (np. 1-2 na klastrze z 20 executorami) oznacza, że większość executorów stoi bezczynnie -- praca nie jest rozłożona równomiernie. Za DUŻO malutkich partycji oznacza narzut na zarządzanie tysiącami maleńkich zadań, który przewyższa realny zysk z równoległości. Dobry punkt startowy w praktyce: 2-4 partycje na rdzeń procesora dostępny w klastrze -- ale to zawsze wymaga eksperymentowania na konkretnych danych.

<a id="sec4"></a>
## 4. Dokładne czytanie planów wykonania

`.explain()` domyślnie pokazuje tylko plan fizyczny. Parametr `mode="formatted"` (albo
`True` dla wszystkich planów) pokazuje WIĘCEJ szczegółów -- łącznie z planem logicznym
przed i po optymalizacji Catalysta.

In [ ]:
zapytanie = (
    zamowienia.join(klienci, "klient_id")
    .filter(F.col("segment") == "Premium")
    .groupBy("miasto")
    .agg(F.count("*").alias("liczba"))
)
zapytanie.explain(mode="formatted")


Klucze do szukania w wypisanym planie:
- `Filter` -- czy filtr trafił jak najbliżej odczytu danych (predicate pushdown)?
- `Project` -- czy Spark czyta tylko potrzebne kolumny (column pruning)?
- `BroadcastHashJoin` / `SortMergeJoin` -- która strategia joina została wybrana (Moduł 4)?
- `Exchange` -- oznacza SHUFFLE w tym miejscu planu -- każde wystąpienie to potencjalny koszt sieciowy.

In [ ]:
zapytanie.explain(mode="cost")


> 🧱 **Spark UI zamiast czytania tekstu**
>
> W praktyce, zwłaszcza w Databricks, dużo wygodniej czyta się te same informacje w graficznym Spark UI (zakładka SQL / DataFrame) niż w tekstowym `.explain()` -- widać tam też CZAS wykonania każdego etapu, a nie tylko sam plan. Tekstowy `.explain()` jest jednak niezastąpiony, gdy pracujesz lokalnie albo chcesz szybko sprawdzić plan bezpośrednio w kodzie/notatniku.

<a id="sec5"></a>
## 5. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- Catalyst i Tungsten -- jak Spark optymalizuje kod DataFrame API/SQL do jednego wspólnego planu fizycznego,
- `cache()`/`persist()` -- zapamiętywanie wyniku dla wielokrotnie używanych DataFrame'ów, i że są leniwe,
- `repartition()` kontra `coalesce()` -- kontrola liczby partycji w pamięci,
- dokładniejsze czytanie `.explain(mode=...)`: `Filter`, `Project`, typ joina, `Exchange` (shuffle).

### 📝 Ćwiczenie 1: Cache dla wielokrotnie używanego DataFrame

Stwórz przefiltrowany DataFrame `zamowienia.filter(status == 'Anulowane')`, wykonaj na nim `cache()`, policz `.count()`, a następnie wykonaj DWIE różne agregacje na tym samym cache'owanym DataFrame.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
anulowane = zamowienia.filter(F.col("status") == "Anulowane").cache()
print(f"Liczba: {anulowane.count()}")

anulowane.groupBy("klient_id").count().orderBy(F.col("count").desc()).show(5)
anulowane.groupBy("pracownik_id").count().orderBy(F.col("count").desc()).show(5)
```

</details>

### 📝 Ćwiczenie 2: repartition kontra coalesce

Sprawdź liczbę partycji `produkty`, zwiększ ją do 6 przez `repartition`, a następnie zmniejsz z powrotem do 2 przez `coalesce` -- porównaj liczby partycji na każdym kroku.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
print(f"Poczatkowo: {produkty.rdd.getNumPartitions()}")
wiecej = produkty.repartition(6)
print(f"Po repartition(6): {wiecej.rdd.getNumPartitions()}")
mniej = wiecej.coalesce(2)
print(f"Po coalesce(2): {mniej.rdd.getNumPartitions()}")
```

</details>

### 📝 Ćwiczenie 3: Szukanie Exchange w planie

Wykonaj join `zamowienia` z `klienci` (bez broadcastu) i wywołaj `.explain(mode="formatted")` -- znajdź w wyniku wystąpienie `Exchange` (shuffle).

<details>
<summary>Pokaż rozwiązanie</summary>

```python
polaczone = zamowienia.join(klienci, "klient_id")
polaczone.explain(mode="formatted")
# Szukaj w wyniku wezlow "Exchange" -- oznaczaja shuffle po obu stronach joina
```

</details>

### 📝 Ćwiczenie 4: Broadcast kontra brak broadcastu -- porównanie planów

Wykonaj TEN SAM join `zamowienia` z `pracownicy` dwa razy: raz bez podpowiedzi, raz z `F.broadcast(pracownicy)`. Porównaj `.explain()` obu wersji -- czy różnią się liczbą węzłów `Exchange`?

<details>
<summary>Pokaż rozwiązanie</summary>

```python
bez_broadcastu = zamowienia.join(pracownicy, "pracownik_id")
z_broadcastem = zamowienia.join(F.broadcast(pracownicy), "pracownik_id")

print("=== Bez broadcastu ===")
bez_broadcastu.explain()
print("=== Z broadcastem ===")
z_broadcastem.explain()
```

</details>

> 🔥 **Wyzwanie: cache w pełnym potoku wieloetapowym**
>
> Zbuduj potok: (1) połącz `zamowienia`, `klienci`, `pozycje_zamowien`, policz `wartosc_pozycji` -- ZACHOWAJ (`cache()`) wynik tego kroku; (2) na cache'owanym wyniku policz TRZY różne raporty: sumę wartości per miasto, sumę wartości per segment, i top 5 klientów po łącznej wartości. Zmierz przybliżony czas wykonania całości przez `time.time()` przed i po -- z cache'em i (dla porównania) bez niego.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
import time

def zbuduj_polaczone():
    return (
        zamowienia.join(klienci, "klient_id")
        .join(pozycje_zamowien, "zamowienie_id")
        .withColumn(
            "wartosc_pozycji",
            F.col("ilosc") * F.col("cena_jednostkowa") * (1 - F.col("rabat")),
        )
    )

# Bez cache -- kazda z trzech akcji przelicza caly lancuch od nowa
start = time.time()
bez_cache = zbuduj_polaczone()
bez_cache.groupBy("miasto").agg(F.sum("wartosc_pozycji")).collect()
bez_cache.groupBy("segment").agg(F.sum("wartosc_pozycji")).collect()
bez_cache.groupBy("klient_id").agg(F.sum("wartosc_pozycji").alias("suma")).orderBy(
    F.col("suma").desc()
).limit(5).collect()
print(f"Bez cache: {time.time() - start:.2f}s")

# Z cache -- lancuch liczy sie raz, potem trzy akcje czytaja z pamieci
start = time.time()
z_cache = zbuduj_polaczone().cache()
z_cache.count()  # wymusza pierwsze wyliczenie i zapisanie w cache'u
z_cache.groupBy("miasto").agg(F.sum("wartosc_pozycji")).collect()
z_cache.groupBy("segment").agg(F.sum("wartosc_pozycji")).collect()
z_cache.groupBy("klient_id").agg(F.sum("wartosc_pozycji").alias("suma")).orderBy(
    F.col("suma").desc()
).limit(5).collect()
print(f"Z cache: {time.time() - start:.2f}s")
```

Na tak malym zestawie danych jak w tym kursie roznica moze byc niewielka albo nawet niewidoczna (narzut samego cache'owania bywa porownywalny z kosztem przeliczenia) -- prawdziwa oszczednosc ujawnia sie na duzych, rozproszonych danych, gdzie ponowne czytanie i przetwarzanie danych zrodlowych jest realnie kosztowne.

</details>

## Co dalej?

To był ostatni moduł czysto techniczny w PySpark. Od **Modułu 10** przechodzimy do
Delta Lake i architektury lakehouse -- fundamentów prawdziwego środowiska Databricks.